# Reproducing Pasquereau & Turner (2013) from dandiset 001636

**Paper.** Pasquereau B, Turner RS (2013). Primary motor cortex of the parkinsonian monkey: altered
neuronal responses to muscle stretch. *Frontiers in Systems Neuroscience* 7:98.
[doi:10.3389/fnsys.2013.00098](https://doi.org/10.3389/fnsys.2013.00098). This attempt used the
full text (open access, PubMed Central PMC3840326) and its supplementary data sheet, whose table
gives the neuronal results of each monkey separately.

**Dandiset.** [DANDI:001636](https://dandiarchive.org/dandiset/001636/0.260518.1819), version
0.260518.1819. The paper (2013) cites no DANDI version. This is the only published version, and
the draft differs from it only by its modification date, so it is the version pinned here. The
dandiset holds single-unit M1 recordings from two monkeys before and after MPTP, from which three
papers were written; this one uses the torque perturbation (muscle stretch) trials.

**What can be tested.** The files of monkey V (Venus, elbow) mark the torque perturbations, with
the joint angle, the torque command and triceps EMG. The files of monkey L (Leu, wrist) mark none,
so every claim is tested on monkey V alone, against the paper's monkey V values where it gives
them (supplementary table) and against its pooled values otherwise.

| Claim | Tested |
|---|---|
| `cell-counts`: 112 and 159 monkey V cells, 49 and 54 PTNs, 31 and 48 CSNs, 3 dual | yes |
| `monkey-l-perturbations`: monkey L torque responses | blocked (data missing) |
| `baseline-rate`: lower resting rate after MPTP in PTNs, not CSNs | yes |
| `ptn-faster-than-csn`: PTNs fire faster than CSNs before MPTP | yes |
| `antidromic-latency`: CSN antidromic latency longer after MPTP, PTN unchanged | yes |
| `false-positive-rate`: the detection threshold's false-positive rate near 0.01 | yes |
| `torque-related-fractions`: monkey V fractions of torque-related cells | yes |
| `prevalence-unchanged`: MPTP does not change the fraction of torque-related cells | yes |
| `ptn-more-responsive`: PTNs respond more often than CSNs | yes |
| `one-direction-half`: 44% of torque-related cells respond to one direction only | yes |
| `excitatory-majority`: most responses are increases; PTN balance unchanged | yes |
| `magnitude-unchanged`: response magnitudes unchanged by MPTP | yes |
| `venus-response-magnitudes`: monkey V response magnitudes | yes |
| `latency-shorter`: response latencies shorter after MPTP | yes |
| `latency-not-training`: latencies do not follow days of training | yes |
| `fwhm-shorter-ptn`: PTN increases briefer after MPTP | yes |
| `dsi-reduced`: fewer directional responses after MPTP | yes |
| `venus-dsi`: monkey V mean DSI | yes |
| `dsi-not-baseline`: resting rate does not differ between directional categories | yes |
| `kinematics-slower`: torque-induced movements slower and smaller after MPTP | yes |
| `kinematic-latency-unchanged`: movement latency unchanged | yes |
| `emg-m2-larger`: larger long-latency (m2) triceps response after MPTP | yes |
| `emg-duration-longer`: longer EMG responses after MPTP | yes |
| `emg-onset-unchanged`: EMG onset unchanged | yes |
| `emg-m1-weak`: m1 much smaller than m2 | yes |
| `velocity-matched-magnitude`: magnitudes unchanged in velocity-matched sessions | yes |
| `session-correlations`: no session-level correlation of PTN latency with velocity or m2 | yes |
| `trial-correlations`: no trial-level correlation | blocked (methods underspecified) |

**Code read.** The paper links no analysis code. The dandiset links its conversion code,
[catalystneuro/turner-lab-to-nwb](https://github.com/catalystneuro/turner-lab-to-nwb), read at
commit e80a192917490b50db6871180b6d1d9a24772e35: the NeuroConv interfaces that built the files,
notes on the source MATLAB files and tutorial notebooks. It settles how the trialized source data
were laid end to end (trials and spikes by the behavioural trial end, analog streams by their
sample count), which explains the analog clock drift found in the data check. It holds no
analysis of the torque responses, so every method choice here comes from the paper's Methods or
is fixed in the claims file.

The claims, with their criteria, are in `claims/001636/10.3389-fnsys.2013.00098.yaml`. The
loading and the measures are in `stretch.py`, whose docstring says which parameters come from the
paper and which were chosen here.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols

import stretch

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
S = stretch.load_all(stretch.asset_urls())
print(f"loaded {len(S)} files of dandiset {stretch.DANDISET_ID} version {stretch.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
sessions = pd.DataFrame([{
    "si": si, "session": s["session_id"], "code": s["session_id"].split("++")[1],
    "subject": s["subject"], "post": s["post"], "date": s["date"], "n_trials": len(s["trials"]),
    "n_flex": int((s["pert_direction"] == "flexion").sum()),
    "n_ext": int((s["pert_direction"] == "extension").sum()), "n_units": len(s["units"]),
    "analog_dt_ms": s.get("analog_dt_ms", np.nan), "has_analog": "windows" in s or "analog_dt_ms" in s,
    "angle_all_nan": s.get("angle_all_nan", True), "torque_all_nan": s.get("torque_all_nan", True),
    "emg_channels": len(s.get("emg_channels", [])), "emg_same_clock": s.get("emg_same_clock", np.nan),
} for si, s in enumerate(S)])
units = pd.DataFrame([{
    "si": si, "ui": ui, "session": s["session_id"], "subject": s["subject"], "post": s["post"],
    "is_post_mptp": u["is_post_mptp"], "projection": u["type"], "ptype": stretch.short_type(u["type"]),
    "unit_name": u["unit_name"], "also_in": u["also_in"], "date": s["date"],
    "n_flex": int((s["pert_direction"] == "flexion").sum()),
    "n_ext": int((s["pert_direction"] == "extension").sum()),
    "antidromic_latency_ms": u["antidromic_latency_ms"], "n_spikes": len(u["spikes"]),
    "isi_below_2ms": u["isi_below_2ms"], "outside_trials": u["outside_trials"], "sorted": u["sorted"],
} for si, s in enumerate(S) for ui, u in enumerate(s["units"])])
units["state"] = np.where(units.post, "post", "pre")
units["studied"] = (units.n_flex >= stretch.MIN_REPS) & (units.n_ext >= stretch.MIN_REPS)

print("Files and units by subject and MPTP state (from the session id):")
print(pd.concat([sessions.groupby(["subject", "post"]).size().rename("files"),
                 units.groupby(["subject", "post"]).size().rename("units")], axis=1).to_string())
print("\nUnits whose is_post_mptp differs from the state in the session id:",
      int((units.post != units.is_post_mptp).sum()))
print("\nRecording dates by subject:")
print(sessions.groupby("subject").date.agg(["min", "max"]).to_string())
print("\nProjection types as stored, by subject and state:")
print(pd.crosstab(units.projection, [units.subject, units.state]).to_string())
print("\nTorque perturbations per file (flexion / extension), by subject and state:")
print(sessions.groupby(["subject", "post"])[["n_flex", "n_ext"]].describe()
      .loc[:, (slice(None), ["min", "50%", "max"])].to_string())
print("\nUnits studied (at least 5 perturbations of each direction), by subject, state and type:")
print(pd.crosstab([units.subject, units.state], [units.studied, units.ptype]).to_string())

The dandiset holds 447 files with 568 units: 210 of monkey L (Leu) and 358 of monkey V (Venus).
The MPTP state in the session id agrees with each unit's `is_post_mptp`. The dandiset's description
says Leu was recorded in 1993; some Leu files are dated 1992.

No monkey L file marks a torque perturbation: `torque_perturbation_type` is "none" in every Leu
trial. The monkey V units with at least 5 perturbations of each direction number 112 before and
159 after MPTP, with 49 and 54 PTNs and 31 and 48 CSNs, the monkey V counts of the paper's
supplementary table. These units are the cells studied here. The other projection types (thalamic
and subthalamic projections) are counted as non-identified (NA) cells, as the paper has no such
category, and the dual PTN and CSN units are kept apart from both.

In [ ]:
hold = []
for s in S:
    if "hold_excursion_deg" not in s["trials"]:
        continue
    tr = s["trials"]
    hold.append(pd.DataFrame({"subject": s["subject"], "post": s["post"], "perturbed": tr.torque != "none",
                              "excursion": tr.hold_excursion_deg, "movement": tr.movement,
                              "active_change": tr.active_angle_change, "end_position": tr.end_position}))
hold = pd.concat(hold, ignore_index=True)
print("Largest angle excursion from its median, 1.0 to 2.5 s after the center target appears (deg):")
print(hold.groupby(["subject", "post", "perturbed"]).excursion.describe(percentiles=[.1, .5, .9]).round(2).to_string())
print("\nFraction of trials with an excursion above 2 deg:")
print(hold.assign(big=hold.excursion > 2).groupby(["subject", "post", "perturbed"]).big.mean().round(3).to_string())
leu = sessions[sessions.subject == "Leu"]
print(f"\nLeu files with no analog data: {int((~leu.has_analog).sum())} of {len(leu)} "
      f"({int((~leu[leu.post].has_analog).sum())} of {int(leu.post.sum())} post-MPTP)")
print("Leu files with analog data whose torque command is all NaN, by state:")
print(leu[leu.has_analog].groupby("post").torque_all_nan.agg(["sum", "count"]).to_string())
print("Analog sample interval (ms) by subject:")
print(sessions.groupby("subject").analog_dt_ms.describe()[["count", "min", "50%", "max"]].round(4).to_string())

In monkey V the joint moves by more than 2 deg in the hold window in 66% (pre) and 86% (post) of
perturbed trials and in 1% and 11% of the others. In monkey L it does so in 15% of trials before
MPTP and 1% after, which does not fit perturbations on two thirds of the trials as the Methods
describe; these windows were set from the monkey V onsets, so the monkey L perturbations could fall
outside them. Either way, the monkey L torque trials cannot be identified from the files, and
`monkey-l-perturbations` is blocked. The Leu torque command is NaN in every pre-MPTP file, and 15
post-MPTP Leu files have no analog data at all. The Leu analog streams are sampled at 100 Hz; the
paper says 200 Hz for monkey L. Monkey V is sampled at 1 kHz (the paper digitized at 500 Hz; the
conversion notes say the lab interpolated to 1 kHz).

In [ ]:
venus = [s for s in S if s["subject"] == "Venus" and "windows" in s]
AX = np.arange(stretch.ANALOG_WINDOW_MS[0], stretch.ANALOG_WINDOW_MS[1] + 1)


def command_onset(w):
    # first ms where the torque command leaves its pre-onset level by more than 6 SD (and 10 units)
    base = np.nanmedian(w[:, :400], axis=1, keepdims=True)
    sd = np.nanstd(w[:, :400], axis=1, keepdims=True)
    dev = np.abs(w - base) > np.maximum(10, 6 * sd)
    return np.where(dev.any(1), AX[np.argmax(dev, axis=1)], np.nan)


realigned = np.concatenate([command_onset(s["windows"]["torque_cmd"]) for s in venus])
stored = np.concatenate([command_onset(s["windows"]["torque_cmd_stored"]) for s in venus])
lag = np.concatenate([s["analog_lag_ms"][s["pert_trial"]] for s in venus])
ok = np.isfinite(stored) & np.isfinite(realigned)
print(f"Monkey V perturbations: {len(realigned)}; command step found in {int(np.isfinite(realigned).sum())}")
print("Torque command step relative to the labelled onset, realigned clock (ms), percentiles 1/50/99:",
      np.nanpercentile(realigned, [1, 50, 99]))
print("Same on the stored analog clock (ms), percentiles 1/50/99:", np.nanpercentile(stored, [1, 50, 99]))
print(f"Stored analog clock behind the trials table at each perturbation: {lag.min():.0f} to {lag.max():.0f} ms; "
      f"correlation of the stored-clock step time with this lag {np.corrcoef(stored[ok], lag[ok])[0, 1]:.3f}")
per_trial = np.concatenate([np.diff(s["analog_lag_ms"]) for s in S if "analog_lag_ms" in s])
print(f"Lag added per trial: median {np.median(per_trial):.1f} ms (range {per_trial.min():.1f} to {per_trial.max():.1f})")
print("EMG series share the angle's timestamps in every monkey V file:",
      bool(sessions[sessions.subject == "Venus"].emg_same_clock.astype(bool).all()))
print("\nStored angle change over active movements (lateral target to reward), median deg:")
print(hold.groupby(["subject", "movement"]).active_change.median().round(1).to_string())
print("derived_end_position, median deg:")
print(hold.groupby(["subject", "movement"]).end_position.median().round(1).to_string())
chg = {d: np.concatenate([s["windows"]["angle"][s["pert_direction"] == d][:, AX == 100][:, 0]
                          - s["windows"]["angle"][s["pert_direction"] == d][:, (AX >= -50) & (AX < 0)].mean(1)
                          for s in venus]) for d in ["flexion", "extension"]}
for d, v in chg.items():
    print(f"Angle change 100 ms after a {d} torque: median {np.nanmedian(v):.2f} deg, positive in "
          f"{np.nanmean(v > 0):.1%} of {np.isfinite(v).sum()} perturbations")

The analog streams (angle, torque command, EMG) are not on the clock of the trials table as
stored. The conversion lays each trial's analog samples end to end with 3 s gaps, using their
sample count, while the trials table and spike times use the behavioural software's trial end,
which is a few ms later. The stored analog clock therefore falls behind by about 5 ms per trial,
reaching hundreds of ms late in a session, and the torque command step appears up to about 100 ms
before the labelled onset (the step time follows the accumulated lag, r = -1.0). With each analog
segment moved back to its trial's start (`stretch.realign`), the command steps 4 to 7 ms after
every labelled onset. All analog measures here use the realigned clock. This is a format error of
the files: the notes say the analog data end 5 ms early but not that later trials are shifted.

The angle's `reference_frame` says positive is extension. The values say the reverse: active
flexion movements raise the angle by about 18 deg and extension movements lower it, in both
monkeys, and flexion torques raise it. The torque labels themselves agree with the EMG (the
triceps, an extensor stretched by a flexion torque, responds to flexion torques; see the EMG
sections). So the reference frame's sign is wrong, and the series are named "Elbow" also for
monkey L, whose task used the wrist.

In [ ]:
# Spikes against the trials table: does the torque direction become decodable from single-cell
# spike counts only after the labelled onset? 20 ms windows, Mann-Whitney flexion vs extension.
edges = np.arange(-200, 201, 20)
studied_v = units[units.studied & (units.subject == "Venus")]
sig = np.zeros(len(edges) - 1)
for r in studied_v.itertuples():
    s, u = S[r.si], S[r.si]["units"][r.ui]
    counts = np.array([np.histogram(x, edges)[0] for x in u["pert"]])
    f, e = counts[s["pert_direction"] == "flexion"], counts[s["pert_direction"] == "extension"]
    for j in range(len(edges) - 1):
        if np.ptp(np.r_[f[:, j], e[:, j]]) > 0:
            sig[j] += stats.mannwhitneyu(f[:, j], e[:, j]).pvalue < 0.01
frac = sig / len(studied_v)
print(f"Fraction of the {len(studied_v)} monkey V cells studied whose spike counts differ between "
      "flexion and extension trials (Mann-Whitney p < 0.01), by window (ms from labelled onset):")
print(pd.Series(frac.round(3), index=[f"{a} to {b}" for a, b in zip(edges[:-1], edges[1:])]).to_string())

Before the labelled onset, the fraction of cells whose counts differ between directions stays near
the 1% expected by chance; it rises in the window from 20 to 40 ms and peaks after. The spike times
therefore share the trials table's clock to within the 20 ms windows, consistent with the 20 to
60 ms response latencies of the paper.

In [ ]:
# Files that repeat another file's recording: same subject and date, and the trial durations of one
# are a contiguous block of the other's.
dur = {si: np.round((s["trials"].stop - s["trials"].start).values, 3) for si, s in enumerate(S)}
repeats = []
for (subj, date), g in sessions.groupby(["subject", "date"]):
    for a in g.si:
        for b in g.si:
            if a != b and len(dur[b]) <= len(dur[a]) and (len(dur[b]) < len(dur[a]) or a < b):
                n = len(dur[b])
                if any(np.array_equal(dur[a][i:i + n], dur[b]) for i in range(len(dur[a]) - n + 1)):
                    repeats.append({"file": sessions.code[a], "repeated_in": sessions.code[b],
                                    "trials": len(dur[a]), "repeated_trials": n,
                                    "units_file": [u["unit_name"] for u in S[a]["units"]],
                                    "units_repeat": [u["unit_name"] for u in S[b]["units"]],
                                    "types_file": [stretch.short_type(u["type"]) for u in S[a]["units"]],
                                    "types_repeat": [stretch.short_type(u["type"]) for u in S[b]["units"]]})
repeats = pd.DataFrame(repeats)
print("Files whose trials repeat a block of another file's trials:")
print(repeats.to_string(index=False))


def trial_spikes(s, u):
    # spike times relative to the start of their trial, one array per trial
    tr = s["trials"]
    k = np.searchsorted(tr.start.values, u["spikes"], side="right") - 1
    return [np.round(u["spikes"][k == i] - tr.start.values[i], 4) for i in range(len(tr))]


same_unit = []
for r in repeats.itertuples():
    a = S[int(sessions[sessions.code == r.file].si.iloc[0])]
    b = S[int(sessions[sessions.code == r.repeated_in].si.iloc[0])]
    da, db = dur[int(sessions[sessions.code == r.file].si.iloc[0])], dur[int(sessions[sessions.code == r.repeated_in].si.iloc[0])]
    off = [i for i in range(len(da) - len(db) + 1) if np.array_equal(da[i:i + len(db)], db)][0]
    for ub in b["units"]:
        tb = trial_spikes(b, ub)
        for ua in a["units"]:
            ta = trial_spikes(a, ua)[off:off + len(db)]
            n_same = sum(np.array_equal(x, y) for x, y in zip(ta, tb))
            same_unit.append({"file": r.file, "unit": ua["unit_name"], "type": ua["type"],
                              "repeat": r.repeated_in, "repeat_unit": ub["unit_name"], "repeat_type": ub["type"],
                              "identical_trials": f"{n_same}/{len(db)}",
                              "spikes_in_block": (sum(map(len, ta)), sum(map(len, tb)))})
same_unit = pd.DataFrame(same_unit)
print("\nUnits of each repeated recording compared trial by trial over the repeated block:")
print(same_unit.to_string(index=False))
dup_units = same_unit[same_unit.identical_trials.str.split("/", regex=False).str[0].astype(int) > 0]
# the copy in the repeated file is the one dropped in the variant that counts each unit once
dup_keys = {(r.repeat, r.repeat_unit) for r in dup_units.itertuples()}
units["duplicate_copy"] = [(sess.split("++")[1], un) in dup_keys for sess, un in zip(units.session, units.unit_name)]
print("Unit copies in repeated files (dropped in the variant that counts each unit once):", sorted(dup_keys))

codes = set(sessions.code)
xref = units[units.also_in != ""].copy()
xref["ref_code"] = xref.also_in.str.split("++", regex=False).str[1]
xref["ref_prefix"] = xref.also_in.str.split("++", regex=False).str[0]
xref["ref_in_dandiset"] = xref.ref_code.isin(codes)
print(f"\nUnits with unit_also_in_session_id: {len(xref)}; prefixes used {xref.ref_prefix.value_counts().to_dict()} "
      f"(session ids use {sorted(sessions.subject.unique())}); referenced file in the dandiset: "
      f"{int(xref.ref_in_dandiset.sum())}")
rows = []
for r in xref[xref.ref_in_dandiset].itertuples():
    other = S[int(sessions[sessions.code == r.ref_code].si.iloc[0])]
    rows.append({"unit": f"{r.session.split('++')[1]}/{r.unit_name}", "type": r.ptype, "spikes": r.n_spikes,
                 "referenced": r.ref_code,
                 "its units": [(u["unit_name"], stretch.short_type(u["type"]), len(u["spikes"])) for u in other["units"]]})
print(pd.DataFrame(rows).to_string(index=False))
print("\nSpike times sorted in every unit:", bool(units["sorted"].all()),
      "| spikes outside the trials:", int(units.outside_trials.sum()))
print("Within-trial ISIs below 2 ms, fraction per unit: median "
      f"{units.isi_below_2ms.median():.4f}, units above 1%: {int((units.isi_below_2ms > 0.01).sum())} of {len(units)} "
      f"({int((units[units.studied & (units.subject == 'Venus')].isi_below_2ms > 0.01).sum())} of the cells studied)")

Four monkey V files repeat a block of trials of another file of the same day: the same recording
(trials, angle, torque, EMG) is deposited twice. In two of them the repeated file's unit is also in
the first file, with identical spikes in every repeated trial, but with another projection type:
v4527 unit 2 is "not_tested" and its copy in v4527b is a PTN; v5811 unit 2 is a PTN and its copy
in v5812 is "no_response". Which type is right cannot be told from the files. The main analysis
keeps every unit as stored, which gives the paper's monkey V counts (so the paper's database may
count these two cells twice); a variant drops the two copies in the repeated files. The repeated
files are left out of the session measures (kinematics, EMG), so that each recording counts once.

The `unit_also_in_session_id` column says the unit was recorded in another session, "identified by
the experimenter based on waveform similarity". Its references use the prefixes "V" and "L", where
the session ids use "Venus" and "Leu", and most referenced files are not in the dandiset. Where the
referenced file is present, it holds a unit with another unit number, another spike count and,
in several cases, another projection type than the referring unit. These are not the same neuron
as stored; the column more likely names the file holding the second unit of the recording. The
units are treated as distinct cells, which also gives the paper's counts.

Every unit's spike times are sorted and fall within the trials. The paper verified isolation by a
refractory period over 2 ms; a few units have more than 1% of their ISIs below 2 ms, and they are
kept, as the dandiset holds the units the paper selected.

In [ ]:
emg_rows = []
for s in venus:
    for ch in stretch.EMG_CHANNELS[:3]:
        if ch not in s["windows"]:
            state = "absent"
        else:
            w = s["windows"][ch]
            if np.isnan(w).any():
                state = "NaN"
            elif np.nanstd(w[:, AX < 0]) < 1:
                state = "flat"
            else:
                state = "used"
        emg_rows.append({"code": s["session_id"].split("++")[1], "post": s["post"], "channel": ch, "state": state})
emg_avail = pd.DataFrame(emg_rows)
emg_avail["repeat"] = emg_avail.code.isin(repeats.repeated_in)
print("Monkey V triceps channels per file (repeated files excluded), by state:")
print(pd.crosstab([emg_avail[~emg_avail.repeat].channel], [emg_avail[~emg_avail.repeat].post, emg_avail[~emg_avail.repeat].state]).to_string())
vals = np.concatenate([s["windows"][ch].ravel() for s in venus for ch in stretch.EMG_CHANNELS[:3] if ch in s["windows"]])
print(f"\nTriceps EMG values (A/D units, as stored in a.u.): {np.nanmin(vals):.0f} to {np.nanmax(vals):.0f}, "
      f"median {np.nanmedian(vals):.0f}")
ang = np.concatenate([s["windows"]["angle"].ravel() for s in venus])
print(f"Monkey V angle around perturbations: {np.nanmin(ang):.1f} to {np.nanmax(ang):.1f} deg")
tq = np.concatenate([s["windows"]["torque_cmd"].ravel() for s in venus])
print(f"Monkey V torque command (arbitrary units): {np.nanmin(tq):.0f} to {np.nanmax(tq):.0f}")
iv = np.concatenate([(s["trials"].onset - s["trials"].center_on).dropna().values for s in venus])
# size of the torque command step (sign-corrected, 10 to 40 ms after onset, less the 100 ms before)
command = []
for s in venus:
    w = s["windows"]["torque_cmd"]
    sign = np.where(s["pert_direction"] == "flexion", 1.0, -1.0)[:, None]
    rel = (w - np.nanmedian(w[:, (AX >= -100) & (AX < 0)], axis=1, keepdims=True)) * sign
    m = np.nanmean(rel, axis=0)
    command.append({"session": s["session_id"], "code": s["session_id"].split("++")[1], "post": s["post"],
                    "date": s["date"], "step": float(np.nanmedian(rel[:, (AX >= 10) & (AX < 40)])),
                    "width_ms": int((m > m.max() / 2).sum()), "after_pulse": float(m[(AX >= 70) & (AX < 150)].mean())})
command = pd.DataFrame(command)
command["level"] = np.round(command.step / command.step[command.step > 10].min()).astype(int)
print("\nTorque command step (units above the pre-onset level) by state; level = step in multiples of the smallest:")
print(pd.crosstab(command.level, command.post).to_string())
print(command.groupby(["post", "level"])[["step", "width_ms", "after_pulse"]].median().round(1).to_string())
print("Dates of the level-2 sessions by state:", command[command.level == 2].groupby("post").date.agg(["min", "max", "count"]).to_string())
print("Torque onset after the center target appears (s), percentiles 0/5/50/95/100:",
      np.percentile(iv, [0, 5, 50, 95, 100]).round(3))

EMG units are A/D values stored as arbitrary units, so EMG is compared only within the dandiset.
The files describe the EMG as rectified and sample-hold integrated over 10 ms; the paper says
rectified and low-pass filtered at 100 Hz.

The torque command differs between the two states. Its 50 ms pulse has three sizes, about 46, 91
and 183 units above the resting level. Most pre-MPTP sessions have the smallest and most post-MPTP
sessions the largest (four times the size), with the middle size in some sessions of each state;
post-MPTP pulses are also followed by an opposite swing of about a quarter of the pulse. The paper
says every impulse was 0.1 Nm for 50 ms. The command is stored in arbitrary units and the files do
not give its scale, so whether the torque or only the command's scaling changed cannot be told
from the files. Either way the pre and post perturbations are not shown to be the same; the
kinematic and EMG claims add a variant restricted to sessions of the middle size, which both
states have.

The torque onset follows the appearance of the center target by 1.70 to 1.73 s in 90% of monkey V
perturbations. The Methods put it 1 to 2 s, uniformly at random, after the monkey captured the
center target. If the column marks the target's appearance, the torque was not unpredictable from
it; the column may instead mark a later event. This does not enter any analysis here.

## Responses of the cells studied

Every monkey V cell studied is analysed once, below, and the claims draw on these tables. For
each cell: the single-trial SDFs (causal gamma kernel, k = 1.5, theta = 20 ms), its baseline rate
(mean SDF over the 500 ms before onset, all perturbations), and for each direction the response
found by the main detection (threshold on the trial-averaged SDF) and by the variant (t-test across
trials), at the true onset and at a pseudo onset 600 ms earlier.

In [ ]:
cells = units[units.studied & (units.subject == "Venus")].copy().reset_index(drop=True)
resp_rows, base = [], []
for c in cells.itertuples():
    s, u = S[c.si], S[c.si]["units"][c.ui]
    X = stretch.sdfs(u["pert"])
    base.append(X[:, stretch._mask(*stretch.BASELINE_MS)].mean())
    for d in ("flexion", "extension"):
        Xd = X[s["pert_direction"] == d]
        for method, shift in (("mean", 0), ("trials", 0), ("mean", stretch.PSEUDO_ONSET_MS),
                              ("trials", stretch.PSEUDO_ONSET_MS)):
            r = stretch.detect(Xd, shift_ms=shift, method=method)
            r.update(cell=c.Index, direction=d, method=method, pseudo=shift != 0)
            resp_rows.append(r)
cells["baseline_rate"] = base
R = pd.DataFrame(resp_rows).merge(cells[["state", "ptype", "si", "date", "duplicate_copy"]],
                                  left_on="cell", right_index=True)
main = R[(R.method == "mean") & ~R.pseudo]
by_cell = main.pivot(index="cell", columns="direction")
cells["torque_related"] = by_cell["responds"].any(axis=1).values
cells["n_directions"] = by_cell["responds"].sum(axis=1).values
cells["latency"] = by_cell["latency"].min(axis=1).values
dsis = [stretch.dsi(main[(main.cell == i) & (main.direction == "flexion")].iloc[0].to_dict(),
                    main[(main.cell == i) & (main.direction == "extension")].iloc[0].to_dict())
        for i in cells.index]
cells["dsi"] = [d for d, _ in dsis]
cells["category"] = [c for _, c in dsis]
resp = main[main.responds].copy()
print(f"{len(cells)} cells studied, {len(main)} cell-directions; responses found: {len(resp)} "
      f"in {int(cells.torque_related.sum())} cells")
print("Cell-directions without a usable baseline (SD of the baseline SDF 0):",
      int((R[(R.method == "mean") & ~R.pseudo].baseline == 0).sum()))
print("Trials per cell and direction: ", main.n_trials.min(), "to", main.n_trials.max(),
      "| baseline rate range", round(cells.baseline_rate.min(), 2), "to", round(cells.baseline_rate.max(), 2), "spikes/s")
print("Response latencies (ms):", resp.latency.min(), "to", resp.latency.max(),
      "| offsets:", resp.offset.min(), "to", resp.offset.max())

### `cell-counts`

> In monkey V, 112 neurons were studied before MPTP and 159 after, of which 49 and 54 were PTNs
> and 31 and 48 CSNs; three neurons in all were activated from both the peduncle and the putamen.

In [ ]:
counts = pd.crosstab(cells.ptype, cells.state).reindex(columns=["pre", "post"])
counts.loc["all"] = counts.sum()
print("Monkey V cells studied by type and state:")
print(counts.to_string())
dual_all = units[units.ptype == "dual"]
print(f"\nUnits activated from both the peduncle and the putamen, all files: {len(dual_all)} "
      f"({dual_all.subject.value_counts().to_dict()}); among the cells studied: {int((cells.ptype == 'dual').sum())}")
print("Other dual projections (with thalamus or STN):", int(units.projection.str.contains('_and_').sum() - len(dual_all)))
print("\nVariant, the two unit copies in repeated files dropped:")
print(pd.crosstab(cells[~cells.duplicate_copy].ptype, cells[~cells.duplicate_copy].state)
      .reindex(columns=["pre", "post"]).to_string())

**Result: reproduced.** The monkey V cells studied number 112 before and 159 after MPTP, with 49 and
54 PTNs and 31 and 48 CSNs, as in the supplementary table, and the files hold 3 units activated
from both the peduncle and the putamen, all in monkey V. Dropping the two unit copies changes the
post-MPTP PTN count to 53.

### `monkey-l-perturbations`

> In monkey L, 57 of 115 neurons before MPTP and 35 of 73 after responded to torque perturbations
> of the wrist.

Blocked: the monkey L files mark no torque perturbation (data check above), so neither the trials
nor their onsets can be found. The files hold 138 and 72 monkey L units, against the paper's 115
and 73 cells studied.

### `baseline-rate`

> Before torque onset, the firing rate of M1 neurons was lower after MPTP, because PTN rates fell
> (Mann-Whitney p < 0.001) while CSN rates did not change (p = 0.5). In monkey V, 12.6 and 8.9
> spikes/s for all cells, 20.3 and 14.8 for PTNs, 3.5 and 3.4 for CSNs.

In [ ]:
def mwu(a, b):
    return stats.mannwhitneyu(a, b, alternative="two-sided").pvalue


expected_rate = {"M1": (12.6, 8.9, 1.98, 1.33), "PTN": (20.3, 14.8, 2.46, 2.72), "CSN": (3.5, 3.4, 1.47, 1.01)}
rows = []
for g, (e_pre, e_post, t_pre, t_post) in expected_rate.items():
    sub = cells if g == "M1" else cells[cells.ptype == g]
    pre, post = sub[sub.state == "pre"].baseline_rate, sub[sub.state == "post"].baseline_rate
    rows.append({"group": g, "n_pre": len(pre), "pre": pre.mean(), "sd_pre": pre.std(), "expected_pre": e_pre,
                 "in_tol_pre": abs(pre.mean() - e_pre) <= t_pre, "n_post": len(post), "post": post.mean(),
                 "sd_post": post.std(), "expected_post": e_post, "in_tol_post": abs(post.mean() - e_post) <= t_post,
                 "change_%": 100 * (post.mean() / pre.mean() - 1), "mwu_p": mwu(pre, post)})
rate_table = pd.DataFrame(rows)
print(rate_table.round(4).to_string(index=False))
nodup = cells[~cells.duplicate_copy]
p = nodup[nodup.ptype == "PTN"]
print(f"\nVariant without the unit copies, PTN: {p[p.state == 'pre'].baseline_rate.mean():.2f} and "
      f"{p[p.state == 'post'].baseline_rate.mean():.2f} spikes/s, p = {mwu(p[p.state == 'pre'].baseline_rate, p[p.state == 'post'].baseline_rate):.4f}")

**Result: reproduced.** All six means are within tolerance (M1 12.42 and 8.87, PTN 18.03 and 14.48,
CSN 3.42 and 3.36 spikes/s). The PTN rate is lower after MPTP (Mann-Whitney p = 0.034) and the CSN
rate does not differ (p = 0.73); the M1 rate is lower (p = 0.0033). The statement's p < 0.001 for
PTNs, from both monkeys, is not reached in monkey V alone. Dropping the unit copies leaves the PTN
result the same (p = 0.034).

### `ptn-faster-than-csn`

> Before MPTP, PTNs fired much faster than CSNs at rest (Mann-Whitney p < 0.001).

In [ ]:
pre = cells[cells.state == "pre"]
ptn_pre, csn_pre = pre[pre.ptype == "PTN"].baseline_rate, pre[pre.ptype == "CSN"].baseline_rate
print(f"PTN {ptn_pre.mean():.2f} spikes/s (range {ptn_pre.min():.2f} to {ptn_pre.max():.2f}, n = {len(ptn_pre)}); "
      f"CSN {csn_pre.mean():.2f} (range {csn_pre.min():.2f} to {csn_pre.max():.2f}, n = {len(csn_pre)}); "
      f"Mann-Whitney p = {mwu(ptn_pre, csn_pre):.2e}")
print("Paper (both monkeys): text 16.3 and 3.6, Table 1 18.3 and 3.2 spikes/s")

**Result: reproduced.** Before MPTP, monkey V PTNs fire at 18.03 spikes/s and CSNs at 3.42
(Mann-Whitney p = 1.3e-11).

### `antidromic-latency`

> PTN antidromic latencies did not change after MPTP (1.9 ms, p = 0.9), while CSN latencies were
> longer (5.1 against 6.1 ms, p < 0.05/3).

In [ ]:
def antidromic(df, label):
    rows = []
    for g, e in (("PTN", (1.9, 1.9, 0.22, 0.2)), ("CSN", (5.1, 6.1, 0.61, 0.89))):
        sub = df[df.ptype == g]
        a, b = sub[sub.state == "pre"].antidromic_latency_ms.dropna(), sub[sub.state == "post"].antidromic_latency_ms.dropna()
        rows.append({"cells": label, "group": g, "n_pre": len(a), "pre": a.mean(), "sd_pre": a.std(),
                     "in_tol_pre": abs(a.mean() - e[0]) <= e[2], "n_post": len(b), "post": b.mean(),
                     "sd_post": b.std(), "in_tol_post": abs(b.mean() - e[1]) <= e[3], "mwu_p": mwu(a, b)})
    return rows


leu_units = units[(units.subject == "Leu") & units.ptype.isin(["PTN", "CSN"])]
ad = pd.DataFrame(antidromic(cells, "monkey V studied")
                  + antidromic(pd.concat([cells, leu_units]), "variant: plus all monkey L units"))
print(ad.round(4).to_string(index=False))
print("Antidromic latency range (ms), cells studied:", cells.antidromic_latency_ms.min(), "to",
      cells.antidromic_latency_ms.max(), "| NaN for", int(cells[cells.ptype.isin(['PTN', 'CSN'])].antidromic_latency_ms.isna().sum()), "PTNs and CSNs")

**Result: partially reproduced.** The four means are within tolerance (PTN 1.96 and 1.86 ms, CSN 4.89
and 6.85 ms) and the PTN latency does not differ (p = 0.54). The CSN latency is longer after MPTP
with p = 0.018, just above the paper's threshold of 0.05/3 = 0.0167. With all monkey L PTNs and
CSNs added (variant), p = 0.0071 for CSNs and 0.91 for PTNs, and the post-MPTP CSN mean (7.01 ms)
is outside tolerance.

### `false-positive-rate`

> The detection threshold, p < 0.01 divided by 5 independent comparisons, holds the chance of a
> false response over the 200 ms epoch near 0.01.

In [ ]:
fp = R[R.pseudo].groupby("method").agg(cell_directions=("responds", "size"),
                                       any_crossing_0_200=("any_crossing_0_200", "mean"),
                                       response_20_60=("responds", "mean"))
real = R[~R.pseudo].groupby("method").agg(any_crossing_0_200=("any_crossing_0_200", "mean"),
                                          response_20_60=("responds", "mean"))
print("At the pseudo onset (600 ms before the torque), fraction of cell-directions:")
print(fp.round(4).to_string())
print("\nAt the true onset, for reference:")
print(real.round(4).to_string())
print("\nPseudo onset, main method, by state:")
print(R[R.pseudo & (R.method == "mean")].groupby("state")[["any_crossing_0_200", "responds"]].mean().round(4).to_string())

In [ ]:
# Variant null: unperturbed trials, at the time a torque usually comes (the session's median delay
# from center target to torque onset), so the epoch matches the true one but has no torque.
null_rows = []
for c in cells.itertuples():
    s, u = S[c.si], S[c.si]["units"][c.ui]
    tr = s["trials"]
    delay = np.nanmedian(tr.onset - tr.center_on)
    t0 = (tr.center_on + delay)[tr.torque == "none"].dropna().values
    if len(t0) < stretch.MIN_REPS:
        continue
    rel = [np.round((u["spikes"][(u["spikes"] >= t - 1.8) & (u["spikes"] < t + 0.7)] - t) * 1000) for t in t0]
    for method in ("mean", "trials"):
        r = stretch.detect(stretch.sdfs(rel), method=method)
        null_rows.append({"cell": c.Index, "method": method, "n_trials": len(t0), "state": c.state,
                          "any_crossing_0_200": r["any_crossing_0_200"], "responds": r["responds"]})
null = pd.DataFrame(null_rows)
print(f"Unperturbed trials at the usual torque time: {null.cell.nunique()} cells with at least 5 such trials "
      f"({null.n_trials.min()} to {null.n_trials.max()} trials)")
print(null.groupby("method")[["any_crossing_0_200", "responds"]].mean().round(4).to_string())

**Result: blocked (methods underspecified).** With the detection fixed in the claims file, a
threshold crossing somewhere in the 200 ms after a pseudo onset 600 ms before the torque occurs in
0.22 of the cell-directions, against the 0.01 the paper's threshold implies; the t-test variant
gives 0.46. Unperturbed trials, taken at the usual torque time, give 0.19 and 0.45. A response
with onset between 20 and 60 ms is found in 0.044 of the cell-directions at the pseudo onset and
0.047 on unperturbed trials (0.37 at the true onset). Both rates are more than 3 times the paper's,
so the procedure here differs from the paper's in a way its Methods do not settle, and the claims
that rest on the selection of responses are blocked with the same reason. Their analyses are run
below all the same, and their numbers are reported.

### `torque-related-fractions`

> In monkey V, 84 of 112 M1 neurons (75%) were torque-related before MPTP and 108 of 159 (68%)
> after; PTNs 27 of 49 (55%) and 37 of 54 (68%); CSNs 14 of 31 (45%) and 10 of 48 (21%).

In [ ]:
expected_frac = {("M1", "pre"): 75, ("M1", "post"): 68, ("PTN", "pre"): 55, ("PTN", "post"): 68,
                 ("CSN", "pre"): 45, ("CSN", "post"): 21}


def related_table(df, label):
    rows = []
    for (g, st), e in expected_frac.items():
        sub = df if g == "M1" else df[df.ptype == g]
        sub = sub[sub.state == st]
        k, n = int(sub.torque_related.sum()), len(sub)
        rows.append({"detection": label, "group": g, "state": st, "related": k, "n": n,
                     "percent": 100 * k / n, "expected": e, "in_tol": abs(100 * k / n - e) <= 0.1 * e})
    return rows


# variant detection: the t-test across trials
var = R[(R.method == "trials") & ~R.pseudo].pivot(index="cell", columns="direction")
cells["torque_related_trials"] = var["responds"].any(axis=1).values
frac_table = pd.DataFrame(related_table(cells, "main")
                          + related_table(cells.assign(torque_related=cells.torque_related_trials), "t-test variant")
                          + related_table(cells[~cells.duplicate_copy], "main, unit copies dropped"))
print(frac_table.round(1).to_string(index=False))

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). The detection here finds 54.5% of monkey V cells
torque-related before MPTP and 47.2% after (paper 75% and 68%), PTNs 71.4% and 66.7% (55% and 68%),
CSNs 16.1% and 16.7% (45% and 21%); only the post-MPTP PTN value is within tolerance. The t-test
variant gives 39.3% and 59.7%.

**Result: blocked (methods underspecified), as the selection is (see `false-positive-rate`).** The
detection here finds 47% of monkey V cells torque-related before MPTP and 52% after (paper 75% and
68%), and the fractions for PTNs and CSNs are also below the paper's except the CSNs after MPTP;
the t-test variant finds more. The numbers are in the table above.

### `prevalence-unchanged`

> MPTP did not change the fraction of torque-related neurons, in M1, in PTNs or in CSNs
> (chi-square, p > 0.05/3).

In [ ]:
def chi2_yates(a_k, a_n, b_k, b_n):
    table = np.array([[a_k, a_n - a_k], [b_k, b_n - b_k]])
    chi2, p, _, _ = stats.chi2_contingency(table, correction=True)
    return chi2, p


rows = []
for label, col in (("main", "torque_related"), ("t-test variant", "torque_related_trials")):
    for g in ("M1", "PTN", "CSN"):
        sub = cells if g == "M1" else cells[cells.ptype == g]
        a, b = sub[sub.state == "pre"][col], sub[sub.state == "post"][col]
        chi2, p = chi2_yates(a.sum(), len(a), b.sum(), len(b))
        rows.append({"detection": label, "group": g, "pre_%": 100 * a.mean(), "post_%": 100 * b.mean(),
                     "change_points": 100 * (b.mean() - a.mean()), "chi2": chi2, "p": p, "p_below_0.05/3": p < 0.05 / 3})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nThe paper's monkey V counts, through the same test:",
      {g: round(chi2_yates(*k)[1], 4) for g, k in {"M1": (84, 112, 108, 159), "PTN": (27, 49, 37, 54), "CSN": (14, 31, 10, 48)}.items()})

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). By the paper's test the fractions do not change in M1 (p = 0.29),
PTNs (p = 0.76) or CSNs (p = 1.0) with the main detection; the t-test variant finds an increase in
M1 (p = 0.0014) and PTNs (p = 0.016), both below 0.05/3.

### `ptn-more-responsive`

> PTNs responded to torque more often than CSNs (68% against 31%, chi-square 32.3, p < 0.001).

In [ ]:
for label, col in (("main", "torque_related"), ("t-test variant", "torque_related_trials")):
    p_, c_ = cells[cells.ptype == "PTN"][col], cells[cells.ptype == "CSN"][col]
    chi2, p = chi2_yates(p_.sum(), len(p_), c_.sum(), len(c_))
    print(f"{label}: PTN {int(p_.sum())}/{len(p_)} ({100 * p_.mean():.1f}%), CSN {int(c_.sum())}/{len(c_)} "
          f"({100 * c_.mean():.1f}%), chi-square {chi2:.1f}, p = {p:.2e}; expected 62.1% and 30.4%")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). PTNs respond more often than CSNs (68.9% against 16.5%,
chi-square 47.5, p = 5.6e-12); the CSN fraction is below the expected 30.4%.

### `one-direction-half`

> About half of the torque-related cells (124 of 284, 44%) responded to only one direction.

In [ ]:
tr_cells = cells[cells.torque_related]
one = (tr_cells.n_directions == 1)
print(f"Torque-related cells responding in one direction only: {int(one.sum())}/{len(tr_cells)} "
      f"({100 * one.mean():.1f}%); by state: {tr_cells.groupby('state').apply(lambda d: round(100 * (d.n_directions == 1).mean(), 1)).to_dict()}")
vt = cells[cells.torque_related_trials]
n_dir_t = var["responds"].sum(axis=1).values[cells.torque_related_trials.values]
print(f"t-test variant: {int((n_dir_t == 1).sum())}/{len(vt)} ({100 * (n_dir_t == 1).mean():.1f}%)")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). 51.5% of torque-related cells (70 of 136) respond to one direction
only, against the expected 44%.

### `excitatory-majority`

> Most torque responses were increases in firing (86% and 88% of M1 responses; 70% and 75% for
> PTNs), and MPTP did not change the PTN balance (chi-square 0.1, p = 0.7).

In [ ]:
rows = []
for g in ("M1", "PTN", "CSN"):
    sub = resp if g == "M1" else resp[resp.ptype == g]
    for st in ("pre", "post"):
        x = sub[sub.state == st]
        rows.append({"group": g, "state": st, "increases": int((x.sign > 0).sum()), "responses": len(x),
                     "percent": 100 * (x.sign > 0).mean(),
                     "flexion": f"{int(((x.sign > 0) & (x.direction == 'flexion')).sum())}/{int((x.direction == 'flexion').sum())}",
                     "extension": f"{int(((x.sign > 0) & (x.direction == 'extension')).sum())}/{int((x.direction == 'extension').sum())}"})
inc = pd.DataFrame(rows)
print(inc.round(1).to_string(index=False))
for g in ("PTN", "CSN"):
    a, b = inc[(inc.group == g) & (inc.state == "pre")].iloc[0], inc[(inc.group == g) & (inc.state == "post")].iloc[0]
    chi2, p = chi2_yates(a.increases, a.responses, b.increases, b.responses)
    print(f"{g} increases pre against post: chi-square {chi2:.2f}, p = {p:.3f}")
print("All responses, both states:", f"{100 * (resp.sign > 0).mean():.1f}% increases of {len(resp)}")
chi2, p = chi2_yates(23, 24, 19, 22)
print(f"Reference, the paper's own CSN counts in Table 1 (23/24 against 19/22): chi-square {chi2:.2f}, p = {p:.2f} "
      "(Table 1 marks this p < 0.01)")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). Increases make up 82.6% and 76.7% of M1 responses and 68.1% and
59.3% of PTN responses (expected 86, 88, 70 and 75%); the PTN balance does not change (chi-square
0.53, p = 0.47). The paper's Table 1 marks its CSN change p < 0.01, but its own counts give p =
0.54 with the test it names.

### `magnitude-unchanged` and `venus-response-magnitudes`

> The magnitude of torque responses did not differ before and after MPTP, for M1 cells or for
> PTNs, by any of three measures (Kolmogorov-Smirnov and Mann-Whitney, all p > 0.15).

> In monkey V the mean response magnitudes were 42.4 and 37.9 spikes/s (M1 flexion), 50.9 and
> 42.6 (M1 extension), 33.4 and 26.1 (PTN flexion), 38.8 and 29.3 (PTN extension).

In [ ]:
measures = ["mean_change", "max_change", "auc"]
rows = []
for g in ("M1", "PTN"):
    sub = resp if g == "M1" else resp[resp.ptype == g]
    for d in ("both", "flexion", "extension"):
        x = sub if d == "both" else sub[sub.direction == d]
        a, b = x[x.state == "pre"], x[x.state == "post"]
        for m in measures:
            rows.append({"group": g, "direction": d, "measure": m, "n_pre": len(a), "n_post": len(b),
                         "mean_pre": a[m].mean(), "mean_post": b[m].mean(),
                         "ks_p": stats.ks_2samp(a[m], b[m]).pvalue, "mwu_p": mwu(a[m], b[m])})
mag = pd.DataFrame(rows)
print(mag.round(4).to_string(index=False))
main_tests = mag[mag.direction == "both"]
print(f"\nMain (directions pooled): smallest p {main_tests[['ks_p', 'mwu_p']].min().min():.4f}; "
      f"tests with p < 0.05: {int((main_tests[['ks_p', 'mwu_p']] < 0.05).sum().sum())} of 12")
for m in measures:
    print(f"{m} range: {resp[m].min():.2f} to {resp[m].max():.2f}")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). Here the responses are smaller after MPTP: 11 of the 12 tests
with directions pooled have p < 0.05 (for example M1 maximum change 43.7 against 32.4 spikes/s,
Kolmogorov-Smirnov p = 0.0017; PTN mean change 23.4 against 15.4, Mann-Whitney p = 0.0001).

In [ ]:
expected_mag = {("M1", "flexion"): (42.4, 37.9), ("M1", "extension"): (50.9, 42.6),
                ("PTN", "flexion"): (33.4, 26.1), ("PTN", "extension"): (38.8, 29.3)}
rows = []
for reading, col in (("a max change", "max_change"), ("b mean rate", "mean_rate"), ("c mean change", "mean_change")):
    within = 0
    for (g, d), (e_pre, e_post) in expected_mag.items():
        sub = resp if g == "M1" else resp[resp.ptype == g]
        x = sub[sub.direction == d]
        for st, e in (("pre", e_pre), ("post", e_post)):
            v = x[x.state == st][col].mean()
            within += abs(v - e) <= 0.1 * e
            rows.append({"reading": reading, "group": g, "direction": d, "state": st, "value": v, "expected": e,
                         "in_tol": abs(v - e) <= 0.1 * e, "n": int((x.state == st).sum())})
mags = pd.DataFrame(rows)
print(mags.round(2).to_string(index=False))
print("\nWithin tolerance per reading (of 8):", mags.groupby("reading").in_tol.sum().to_dict())

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). The maximum change matches the four pre-MPTP values and
none of the post-MPTP ones (4 of 8 within tolerance); the other readings match 1 and 0 of 8.

### `latency-shorter`

> Response latencies were shorter after MPTP for M1 cells and for PTNs, for flexion and extension
> (Kolmogorov-Smirnov, all p < 0.05). Monkey V means 39.2 and 34.6 ms (M1 flexion), 40.4 and 33.4
> (M1 extension), 43.9 and 35.5 (PTN flexion), 44.4 and 36.7 (PTN extension).

In [ ]:
expected_lat = {("M1", "flexion"): (39.2, 34.56), ("M1", "extension"): (40.4, 33.4),
                ("PTN", "flexion"): (43.9, 35.5), ("PTN", "extension"): (44.4, 36.7)}


def latency_table(rsp):
    rows = []
    for (g, d), (e_pre, e_post) in expected_lat.items():
        sub = rsp if g == "M1" else rsp[rsp.ptype == g]
        x = sub[sub.direction == d]
        a, b = x[x.state == "pre"].latency, x[x.state == "post"].latency
        rows.append({"group": g, "direction": d, "n_pre": len(a), "pre": a.mean(), "expected_pre": e_pre,
                     "in_tol_pre": abs(a.mean() - e_pre) <= 0.1 * e_pre, "n_post": len(b), "post": b.mean(),
                     "expected_post": e_post, "in_tol_post": abs(b.mean() - e_post) <= 0.1 * e_post,
                     "ks_p": stats.ks_2samp(a, b).pvalue, "mwu_p": mwu(a, b)})
    return pd.DataFrame(rows)


lat = latency_table(resp)
print(lat.round(4).to_string(index=False))
print("\nPooled means (paper, both monkeys: M1 41 and 35 ms, PTN 46 and 38 ms):")
for g in ("M1", "PTN"):
    sub = resp if g == "M1" else resp[resp.ptype == g]
    cs = cells[cells.torque_related] if g == "M1" else cells[cells.torque_related & (cells.ptype == g)]
    print(f"  {g}: responses {sub[sub.state == 'pre'].latency.mean():.1f} and {sub[sub.state == 'post'].latency.mean():.1f} ms; "
          f"earliest per cell {cs[cs.state == 'pre'].latency.mean():.1f} and {cs[cs.state == 'post'].latency.mean():.1f} ms "
          f"(KS p = {stats.ks_2samp(cs[cs.state == 'pre'].latency, cs[cs.state == 'post'].latency).pvalue:.4f})")
resp_t = R[(R.method == "trials") & ~R.pseudo & R.responds]
print("\nt-test variant:")
print(latency_table(resp_t)[["group", "direction", "n_pre", "pre", "n_post", "post", "ks_p"]].round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, g in zip(axes, ("M1", "PTN")):
    sub = resp if g == "M1" else resp[resp.ptype == g]
    for st, col in (("pre", "k"), ("post", "0.6")):
        for d, ls in (("flexion", "-"), ("extension", "--")):
            v = np.sort(sub[(sub.state == st) & (sub.direction == d)].latency)
            ax.step(v, np.arange(1, len(v) + 1) / len(v), color=col, ls=ls, where="post", label=f"{st} {d}")
    ax.set_title(g)
    ax.set_xlabel("response onset (ms)")
axes[0].set_ylabel("cumulative fraction")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()
print("Plotted latencies span", resp.latency.min(), "to", resp.latency.max(), "ms; none outside the axes.")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). With the main detection, latencies are shorter after MPTP in all four
comparisons (Kolmogorov-Smirnov p = 0.0086, 0.0048, 0.022 and 0.0023) and all eight means are
within tolerance (for example M1 flexion 40.8 and 35.9 ms, PTN extension 46.6 and 37.8 ms). The
t-test variant finds no significant shortening (p 0.27 to 0.87).

### `latency-not-training`

> Latencies did not correlate with the days of training (monkey V Spearman |rho| = 0.02, p = 0.85).

In [ ]:
tr_cells = cells[cells.torque_related].copy()
first_day = sessions[sessions.subject == "Venus"].date.min()
tr_cells["day"] = (tr_cells.date - first_day).dt.days
rho, p = stats.spearmanr(tr_cells.day, tr_cells.latency)
print(f"All torque-related cells (n = {len(tr_cells)}): Spearman rho = {rho:.3f}, p = {p:.4f}; days {tr_cells.day.min()} to {tr_cells.day.max()}")
for st in ("pre", "post"):
    x = tr_cells[tr_cells.state == st]
    rho, p = stats.spearmanr(x.day, x.latency)
    print(f"  {st}: n = {len(x)}, rho = {rho:.3f}, p = {p:.4f}")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). Latency correlates with the day of recording over both states
(rho = -0.26, p = 0.0022), as post-MPTP sessions come later; within each state it does not
(p = 0.42 and 0.21).

### `fwhm-shorter-ptn`

> Torque-evoked increases in PTN firing were briefer after MPTP (FWHM 73 to 50 ms over both
> monkeys; Mann-Whitney p < 0.05 for each direction). Monkey V: 97 and 52 ms (flexion), 57 and
> 41 ms (extension).

In [ ]:
expected_fwhm = {"flexion": (97, 52), "extension": (57, 41)}
ptn = resp[resp.ptype == "PTN"]
rows = []
for subset, x in (("increases", ptn[ptn.sign > 0]), ("all PTN responses", ptn)):
    for d, (e_pre, e_post) in expected_fwhm.items():
        y = x[x.direction == d]
        a, b = y[y.state == "pre"].fwhm, y[y.state == "post"].fwhm
        rows.append({"responses": subset, "direction": d, "n_pre": len(a), "pre": a.mean(),
                     "in_tol_pre": abs(a.mean() - e_pre) <= 0.1 * e_pre, "n_post": len(b), "post": b.mean(),
                     "in_tol_post": abs(b.mean() - e_post) <= 0.1 * e_post, "mwu_p": mwu(a, b)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
inc = ptn[ptn.sign > 0]
print(f"\nPTN increases, directions pooled: {inc[inc.state == 'pre'].fwhm.mean():.1f} and {inc[inc.state == 'post'].fwhm.mean():.1f} ms "
      f"(paper 73 and 50), p = {mwu(inc[inc.state == 'pre'].fwhm, inc[inc.state == 'post'].fwhm):.4f}; FWHM range {ptn.fwhm.min():.0f} to {ptn.fwhm.max():.0f} ms")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). PTN increases are briefer after MPTP for flexion (93.8 against 40.8
ms, p = 0.039) but not for extension (56.9 against 41.6 ms, p = 0.39).

### `dsi-reduced`, `venus-dsi` and `dsi-not-baseline`

> Directional selectivity fell after MPTP; reciprocal and unidirectional responses became less
> common (chi-square 22.2, p < 0.001).

> In monkey V the mean DSI was 0.72 and 0.66 (M1), 0.73 and 0.75 (PTN), 0.81 and 0.83 (CSN).

> Pre-torque rates did not differ between directional categories (one-way ANOVA, F = 0.77, p = 0.46).

In [ ]:
cats = ["reciprocal", "unidirectional", "bidirectional", "non-directional"]
tr_cells = cells[cells.torque_related]
tab = pd.crosstab(tr_cells.category, tr_cells.state).reindex(index=cats, columns=["pre", "post"]).fillna(0)
print(tab.to_string())
chi2, p, dof, _ = stats.chi2_contingency(tab.values)
high = tr_cells.category.isin(["reciprocal", "unidirectional"])
fr = high.groupby(tr_cells.state).mean()
print(f"\nReciprocal or unidirectional: pre {100 * fr['pre']:.1f}%, post {100 * fr['post']:.1f}% "
      f"(change {100 * (fr['post'] - fr['pre']):.1f} points); chi-square (2 x 4) {chi2:.2f}, dof {dof}, p = {p:.4f}")
for g in ("PTN", "CSN"):
    x = tr_cells[tr_cells.ptype == g]
    t = pd.crosstab(x.category, x.state).reindex(index=cats, columns=["pre", "post"]).fillna(0)
    t = t[t.sum(axis=1) > 0]
    c2, pp, _, _ = stats.chi2_contingency(t.values)
    h = x.category.isin(["reciprocal", "unidirectional"]).groupby(x.state).mean()
    print(f"  {g}: reciprocal or unidirectional pre {100 * h.get('pre', np.nan):.1f}%, post {100 * h.get('post', np.nan):.1f}%; chi-square p = {pp:.3f}")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). Reciprocal and unidirectional cells fall from 59.0% to 52.0% of
torque-related cells, by 7.0 points, without significance (chi-square 6.26, p = 0.10).

In [ ]:
expected_dsi = {("M1", "pre"): (0.72, 0.087), ("M1", "post"): (0.66, 0.077), ("PTN", "pre"): (0.73, 0.115),
                ("PTN", "post"): (0.75, 0.132), ("CSN", "pre"): (0.81, 0.214), ("CSN", "post"): (0.83, 0.253)}
rows = []
for (g, st), (e, tol) in expected_dsi.items():
    x = tr_cells if g == "M1" else tr_cells[tr_cells.ptype == g]
    v = x[x.state == st].dsi
    rows.append({"group": g, "state": st, "n": len(v), "mean_dsi": v.mean(), "sd": v.std(), "expected": e,
                 "in_tol": abs(v.mean() - e) <= tol})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("DSI range:", round(tr_cells.dsi.min(), 3), "to", round(tr_cells.dsi.max(), 3))

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). Three of the six mean DSIs are within tolerance (PTN pre and both CSN
values); M1 is 0.63 and 0.48 and PTN post 0.50, against 0.72, 0.66 and 0.75.

In [ ]:
groups = [tr_cells[tr_cells.category == c].baseline_rate for c in cats]
f4, p4 = stats.f_oneway(*groups)
merged = tr_cells.category.replace({"reciprocal": "highly directional", "unidirectional": "highly directional"})
f3, p3 = stats.f_oneway(*[tr_cells[merged == c].baseline_rate for c in merged.unique()])
print("Baseline rate by category (mean, n):", {c: (round(g.mean(), 2), len(g)) for c, g in zip(cats, groups)})
print(f"One-way ANOVA, four categories: F = {f4:.2f}, p = {p4:.4f}; three groups: F = {f3:.2f}, p = {p3:.4f}")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). Baseline rates do not differ between the four categories (F = 0.58,
p = 0.63).

## Kinematics and EMG

Session measures use monkey V files with analog data, leaving out the four files that repeat
another file's recording. Kinematics are measured on every perturbation; a trial whose joint was
already moving faster than 5 deg/s in the 50 ms before onset is counted and kept.

In [ ]:
kin_rows = []
for s in venus:
    if s["session_id"].split("++")[1] in set(repeats.repeated_in):
        continue
    for k, d in enumerate(s["pert_direction"]):
        r = stretch.kinematics(s["windows"]["angle"][k], d, s["analog_step_ms"])
        if r["valid"]:
            kin_rows.append({"session": s["session_id"], "post": s["post"], "direction": d, **r})
kin = pd.DataFrame(kin_rows)
kin["state"] = np.where(kin.post, "post", "pre")
print(f"{len(kin)} perturbations in {kin.session.nunique()} sessions; moving before onset: {int(kin.moving_before.sum())}; "
      f"no velocity crossing within 150 ms: {int(kin.latency_ms.isna().sum())}")
print("Ranges: latency", kin.latency_ms.min(), "to", kin.latency_ms.max(), "ms; peak velocity",
      round(kin.peak_velocity.min(), 1), "to", round(kin.peak_velocity.max(), 1), "deg/s; amplitude",
      round(kin.amplitude.min(), 2), "to", round(kin.amplitude.max(), 2), "deg")
sess_kin = kin.groupby(["session", "state", "direction"])[["latency_ms", "peak_velocity", "amplitude"]].mean().reset_index()
sess_kin["level"] = sess_kin.session.map(command.set_index("session").level)
print("\nSession means, mean over sessions:")
print(sess_kin.groupby(["direction", "state"])[["latency_ms", "peak_velocity", "amplitude"]].agg(["mean", "count"]).round(2).to_string())

### `kinematics-slower`

> After MPTP, torque-induced displacements were slower and smaller; mean peak velocity fell by
> 16.2% and amplitude by 6.6% (ANOVA, F > 515 and F > 49).

In [ ]:
def anova(df, measure):
    model = ols(f"{measure} ~ C(state) * C(direction)", data=df).fit()
    return sm.stats.anova_lm(model, typ=2)


for m in ("peak_velocity", "amplitude"):
    print(f"{m}:")
    print(anova(sess_kin, m).round(4).to_string())
    print()
rows = []
for reading, x in (("a both directions", sess_kin), ("b flexion", sess_kin[sess_kin.direction == "flexion"]),
                   ("extension, for reference", sess_kin[sess_kin.direction == "extension"])):
    g = x.groupby("state")[["peak_velocity", "amplitude"]].mean()
    rows.append({"reading": reading, "velocity_change_%": 100 * (g.loc["post", "peak_velocity"] / g.loc["pre", "peak_velocity"] - 1),
                 "amplitude_change_%": 100 * (g.loc["post", "amplitude"] / g.loc["pre", "amplitude"] - 1)})
kin_change = pd.DataFrame(rows)
kin_change["velocity_in_tol"] = (kin_change["velocity_change_%"] + 16.2).abs() <= 1.62
kin_change["amplitude_in_tol"] = (kin_change["amplitude_change_%"] + 6.6).abs() <= 0.66
print(kin_change.round(2).to_string(index=False))
lvl2 = sess_kin[sess_kin.level == 2]
g = lvl2.groupby("state")[["peak_velocity", "amplitude"]].mean()
print(f"\nVariant, sessions with the middle command size only ({lvl2[lvl2.state == 'pre'].session.nunique()} pre, "
      f"{lvl2[lvl2.state == 'post'].session.nunique()} post): peak velocity {g.loc['pre', 'peak_velocity']:.1f} and "
      f"{g.loc['post', 'peak_velocity']:.1f} deg/s ({100 * (g.loc['post', 'peak_velocity'] / g.loc['pre', 'peak_velocity'] - 1):.1f}%), "
      f"amplitude {g.loc['pre', 'amplitude']:.2f} and {g.loc['post', 'amplitude']:.2f} deg "
      f"({100 * (g.loc['post', 'amplitude'] / g.loc['pre', 'amplitude'] - 1):.1f}%)")
for m in ("peak_velocity", "amplitude"):
    print(f"  {m}: MPTP F = {anova(lvl2, m).loc['C(state)', 'F']:.2f}, p = {anova(lvl2, m).loc['C(state)', 'PR(>F)']:.4f}")
print("\nBy command size (session means, both directions):")
print(sess_kin.groupby(["level", "state"])[["peak_velocity", "amplitude", "latency_ms"]].agg(["mean", "count"]).round(1).to_string())
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, m in zip(axes, ("latency_ms", "peak_velocity", "amplitude")):
    for i, (d, st) in enumerate([(d, st) for d in ("flexion", "extension") for st in ("pre", "post")]):
        v = sess_kin[(sess_kin.direction == d) & (sess_kin.state == st)][m]
        ax.scatter(np.full(len(v), i) + np.random.default_rng(0).uniform(-0.15, 0.15, len(v)), v, s=6,
                   color="k" if st == "pre" else "0.6")
    ax.set_xticks(range(4), ["flex pre", "flex post", "ext pre", "ext post"], fontsize=7)
    ax.set_title(m)
plt.tight_layout()
plt.show()
print("All session means are plotted (ranges printed above).")

**Result: not reproduced.** In monkey V, torque-induced movements are faster and larger after
MPTP: session-mean peak velocity rises by 28.8% and amplitude by 15.2% over both directions
(30.3% and 12.0% for flexion), with MPTP effects F = 103 and 26 (p < 0.0001). Restricted to the
sessions with the same torque command size (variant), the increase is larger (101.9% and 113.4%).
The velocities here (session means 8.5 to 145.3 deg/s) are also lower than the paper's 130 to
200 deg/s range of matched sessions.

### `kinematic-latency-unchanged`

> The latency of torque-induced displacements did not change after MPTP (F < 0.2, p > 0.05).

In [ ]:
print(anova(sess_kin, "latency_ms").round(4).to_string())
print(sess_kin.groupby(["direction", "state"]).latency_ms.mean().round(2).to_string())
a = anova(lvl2, "latency_ms")
print(f"Variant, middle command size only: MPTP F = {a.loc['C(state)', 'F']:.2f}, p = {a.loc['C(state)', 'PR(>F)']:.4f}; "
      f"means {lvl2.groupby('state').latency_ms.mean().round(2).to_dict()}")

**Result: not reproduced.** The movement onset latency is shorter after MPTP (session means 9.8 and
3.2 ms for flexion, 11.0 and 6.0 ms for extension; MPTP F = 1032, p < 0.0001), with an MPTP by
direction interaction (F = 19.1). In the sessions with the same command size (variant) it does not
differ (7.81 and 7.73 ms, p = 0.79).

### EMG

Session-average responses of the three triceps channels to flexion torques, which stretch the
triceps, with the extension torques shown for the contrast.

In [ ]:
emg_rows, traces = [], {}
for s in venus:
    code_ = s["session_id"].split("++")[1]
    if code_ in set(repeats.repeated_in):
        continue
    for ch in stretch.EMG_CHANNELS[:3]:
        if ch not in s["windows"]:
            continue
        w = s["windows"][ch]
        if np.isnan(w).any() or np.nanstd(w[:, AX < 0]) < 1:
            continue
        for d in ("flexion", "extension"):
            r = stretch.emg_response(w[s["pert_direction"] == d], s["analog_step_ms"])
            emg_rows.append({"session": code_, "post": s["post"], "channel": ch, "direction": d,
                             **{k: r[k] for k in ("m1", "m2", "onset_ms", "duration_ms")}})
            traces[(code_, ch, d)] = r["trace"]
emg = pd.DataFrame(emg_rows)
emg["state"] = np.where(emg.post, "post", "pre")
emg["level"] = emg.session.map(command.set_index("code").level)
print("Channel-direction-sessions without perturbations of that direction (dropped):", int(emg.m2.isna().sum()))
emg = emg.dropna(subset=["m2"])
print("Channel-sessions used:")
print(pd.crosstab([emg.channel, emg.direction], emg.state).to_string())
print("\nMedian m2 (A/D units) by channel, direction and state:")
print(emg.groupby(["channel", "direction", "state"]).m2.median().unstack().round(2).to_string())
print("\nRanges: m1", round(emg.m1.min(), 1), "to", round(emg.m1.max(), 1), "; m2", round(emg.m2.min(), 1), "to",
      round(emg.m2.max(), 1), "; responses found in", int(emg.onset_ms.notna().sum()), "of", len(emg))
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharex=True)
for ax, ch in zip(axes, stretch.EMG_CHANNELS[:3]):
    for st, col in (("pre", "k"), ("post", "0.6")):
        keys = [(r.session, ch, "flexion") for r in emg[(emg.channel == ch) & (emg.state == st) & (emg.direction == "flexion")].itertuples()]
        if keys:
            m = np.mean([traces[k] for k in keys], axis=0)
            ax.plot(AX, m, color=col, label=f"{st} (n = {len(keys)})")
    ax.set_xlim(-50, 200)
    ax.axvspan(41, 80, color="0.9")
    ax.set_title(f"{ch}, flexion torque")
    ax.set_xlabel("ms from torque onset")
    ax.legend(fontsize=7)
axes[0].set_ylabel("EMG less baseline (A/D units)")
plt.tight_layout()
plt.show()
print("Grand averages are plotted from -50 to 200 ms (shaded: m2 window); y range is set by the data.")

### `emg-m2-larger`

> The long-latency (m2, 41 to 80 ms) EMG response of the stretched triceps was larger after MPTP
> (Kolmogorov-Smirnov, p < 0.05).

In [ ]:
rows = []
flex = emg[emg.direction == "flexion"]
for ch in stretch.EMG_CHANNELS[:3]:
    x = flex[flex.channel == ch]
    a, b = x[x.state == "pre"].m2, x[x.state == "post"].m2
    rows.append({"channel": ch, "n_pre": len(a), "median_pre": a.median(), "mean_pre": a.mean(), "n_post": len(b),
                 "median_post": b.median(), "mean_post": b.mean(), "ks_p": stats.ks_2samp(a, b).pvalue,
                 "larger_and_p<0.05": (b.median() > a.median()) and stats.ks_2samp(a, b).pvalue < 0.05})
m2_table = pd.DataFrame(rows)
print(m2_table.round({c: 2 for c in ("median_pre", "mean_pre", "median_post", "mean_post")})
      .to_string(index=False, formatters={"ks_p": "{:.2e}".format}))
print("\nVariant, middle command size only:")
for ch in stretch.EMG_CHANNELS[:3]:
    x = flex[(flex.channel == ch) & (flex.level == 2)]
    a, b = x[x.state == "pre"].m2, x[x.state == "post"].m2
    if len(a) and len(b):
        print(f"  {ch}: n {len(a)} and {len(b)}, median m2 {a.median():.2f} and {b.median():.2f}, KS p = {stats.ks_2samp(a, b).pvalue:.4f}")
print("\nMedian m2 by command size and state (flexion, all triceps channels):")
print(flex.groupby(["level", "state"]).m2.agg(["median", "count"]).round(2).to_string())

**Result: reproduced.** For flexion torques, the m2 response is larger after MPTP in all three
triceps channels (medians 3.0 against 164.2, 3.1 against 297.5 and 5.2 against 76.2 A/D units;
Kolmogorov-Smirnov p < 1e-10 for each). The increase follows the torque command: in sessions of the
middle command size (variant), m2 is not larger after MPTP (triceps longus 1 median 46.7 against
1.3, p = 0.013, in the opposite direction), and the median m2 is 0.61 at the smallest command size,
40.8 at the middle and 194.6 at the largest.

### `emg-duration-longer` and `emg-onset-unchanged`

> The total duration of EMG responses to torque increased after MPTP, from 57 +/- 6 to 79 +/- 3 ms
> (Kolmogorov-Smirnov, p < 0.001). The latency of the earliest EMG responses did not change.

In [ ]:
fr = flex.dropna(subset=["duration_ms"])
a, b = fr[fr.state == "pre"], fr[fr.state == "post"]
print(f"Responses found: pre {len(a)} of {int((flex.state == 'pre').sum())}, post {len(b)} of {int((flex.state == 'post').sum())} channel-sessions")
print(f"Duration: pre {a.duration_ms.mean():.1f} ms (SEM {a.duration_ms.sem():.1f}, median {a.duration_ms.median():.0f}), "
      f"post {b.duration_ms.mean():.1f} ms (SEM {b.duration_ms.sem():.1f}, median {b.duration_ms.median():.0f}); "
      f"KS p = {stats.ks_2samp(a.duration_ms, b.duration_ms).pvalue:.4g}; range {fr.duration_ms.min():.0f} to {fr.duration_ms.max():.0f} ms")
print(f"Onset: pre {a.onset_ms.mean():.1f} ms, post {b.onset_ms.mean():.1f} ms; KS p = {stats.ks_2samp(a.onset_ms, b.onset_ms).pvalue:.4g}; "
      f"range {fr.onset_ms.min():.0f} to {fr.onset_ms.max():.0f} ms")
l2 = fr[fr.level == 2]
print(f"Variant, middle command size only: duration {l2[l2.state == 'pre'].duration_ms.mean():.1f} and "
      f"{l2[l2.state == 'post'].duration_ms.mean():.1f} ms (n {int((l2.state == 'pre').sum())} and {int((l2.state == 'post').sum())}); "
      f"onset {l2[l2.state == 'pre'].onset_ms.mean():.1f} and {l2[l2.state == 'post'].onset_ms.mean():.1f} ms; "
      f"KS p duration {stats.ks_2samp(l2[l2.state == 'pre'].duration_ms, l2[l2.state == 'post'].duration_ms).pvalue:.4f}, "
      f"onset {stats.ks_2samp(l2[l2.state == 'pre'].onset_ms, l2[l2.state == 'post'].onset_ms).pvalue:.4f}")
print("\nBy channel:")
print(fr.groupby(["channel", "state"])[["duration_ms", "onset_ms"]].agg(["mean", "count"]).round(1).to_string())
print("\nThreshold variants (pooled channels):")
for sd in (1.5, 3.0):
    rr = []
    post_of = dict(zip(sessions.code, sessions.post))
    for k, tr in traces.items():
        if k[2] != "flexion":
            continue
        st = "post" if post_of[k[0]] else "pre"
        b0 = (AX >= -200) & (AX < 0)
        thr = sd * tr[b0].std(ddof=1)
        above = (tr > thr) & (AX >= 10)
        cand = np.flatnonzero(above & (AX <= 150))
        if len(cand):
            rest = np.flatnonzero(~above[cand[0]:])
            rr.append({"state": st, "duration": int(rest[0]) if len(rest) else np.nan, "onset": AX[cand[0]]})
    rr = pd.DataFrame(rr).dropna()
    pa, pb = rr[rr.state == "pre"], rr[rr.state == "post"]
    print(f"  {sd} SD: duration {pa.duration.mean():.1f} and {pb.duration.mean():.1f} ms (KS p = {stats.ks_2samp(pa.duration, pb.duration).pvalue:.4g}); "
          f"onset {pa.onset.mean():.1f} and {pb.onset.mean():.1f} ms (KS p = {stats.ks_2samp(pa.onset, pb.onset).pvalue:.4g})")

**Result: partially reproduced.** EMG responses last longer after MPTP (24.5 against 50.4 ms,
Kolmogorov-Smirnov p = 1.8e-39), but both means are outside tolerance (expected 57 and 79 ms);
thresholds of 1.5 and 3 SD give the same direction (16.4 against 48.5
and 17.3 against 49.5 ms). At the middle command size (variant) the durations do
not differ (28.4 and 27.9 ms, p = 0.96).

**Result: not reproduced.** The onset of the EMG response is earlier after MPTP (40.8 against
24.0 ms, Kolmogorov-Smirnov p = 5.5e-21), with the same direction at thresholds of 1.5 and 3 SD.
Pre-MPTP responses are small (median m2 3.0 to 5.2 A/D units), which delays their detected onset.
At the middle command size (variant) the onsets do not differ (p = 0.18).

### `emg-m1-weak`

> The short-latency m1 component (15 to 40 ms) was poorly formed and generally much smaller than m2.

In [ ]:
for st in ("pre", "post"):
    x = flex[flex.state == st]
    ratio = x.m1 / x.m2
    print(f"{st}: m2 > m1 in {100 * (x.m2 > x.m1).mean():.1f}% of {len(x)} channel-sessions; median m1 {x.m1.median():.2f}, "
          f"median m2 {x.m2.median():.2f}, median m1/m2 {ratio.median():.2f} (over channel-sessions with m2 > 0: "
          f"{(x.m1 / x.m2)[x.m2 > 0].median():.2f})")

**Result: partially reproduced.** After MPTP m2 exceeds m1 in 96.5% of channel-sessions with a
median m1 / m2 of 0.22. Before MPTP the median ratio is 0.10, but m2 exceeds m1 in 73.5%, just
below the 75% of the criterion.

### `velocity-matched-magnitude`

> In sessions with equivalent torque-induced velocities (130 to 200 deg/s), MPTP did not change the
> magnitude of responses (Mann-Whitney, p = 0.19 and 0.46; 24 and 33 flexion, 22 and 31 extension
> responses).

In [ ]:
sv = sess_kin.set_index(["session", "direction"]).peak_velocity
resp_v = resp.copy()
resp_v["session"] = [S[si]["session_id"] for si in resp_v.si]
resp_v["session_velocity"] = [sv.get((s, d), np.nan) for s, d in zip(resp_v.session, resp_v.direction)]
print("Responses without a session velocity (repeated files):", int(resp_v.session_velocity.isna().sum()))
matched = resp_v[(resp_v.session_velocity >= 130) & (resp_v.session_velocity <= 200)]
rows = []
for d in ("flexion", "extension"):
    x = matched[matched.direction == d]
    a, b = x[x.state == "pre"], x[x.state == "post"]
    row = {"direction": d, "n_pre": len(a), "n_post": len(b),
           "velocity_pre": a.session_velocity.mean(), "velocity_post": b.session_velocity.mean(),
           "velocity_p": mwu(a.session_velocity, b.session_velocity)}
    for m in ("max_change", "mean_change", "auc"):
        row[f"{m}_pre"], row[f"{m}_post"], row[f"{m}_p"] = a[m].mean(), b[m].mean(), mwu(a[m], b[m])
    rows.append(row)
print(pd.DataFrame(rows).round(4).T.to_string())
print("Session mean peak velocity range:", round(sess_kin.peak_velocity.min(), 1), "to", round(sess_kin.peak_velocity.max(), 1), "deg/s")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). It is also blocked by the velocity scale: only one
response comes from a session whose mean peak velocity is between 130 and 200 deg/s (session means
reach 145.3 deg/s here), against 24 to 33 per group in the paper.

### `session-correlations`

> Across sessions, the mean PTN latency did not correlate with peak velocity (|rho| < 0.18) or with
> the m2 EMG response (|rho| < 0.26; Spearman, threshold 0.05/4).

In [ ]:
ptn_cells = cells[cells.torque_related & (cells.ptype == "PTN")].copy()
ptn_cells["code"] = [S[si]["session_id"].split("++")[1] for si in ptn_cells.si]
ptn_cells["session"] = [S[si]["session_id"] for si in ptn_cells.si]
per_session = ptn_cells.groupby(["session", "code"]).latency.mean().reset_index()
vel = sess_kin.groupby("session").peak_velocity.mean()
m2 = flex[flex.channel == "EMGTricepsLongus1"].set_index("session").m2
per_session["velocity"] = per_session.session.map(vel)
per_session["m2"] = per_session.code.map(m2)
print(f"Sessions with a torque-related PTN: {len(per_session)}; with velocity {int(per_session.velocity.notna().sum())}; "
      f"with triceps longus 1 m2 {int(per_session.m2.notna().sum())}")
for col in ("velocity", "m2"):
    x = per_session.dropna(subset=[col])
    rho, p = stats.spearmanr(x.latency, x[col])
    print(f"PTN latency against {col}: Spearman rho = {rho:.3f}, p = {p:.4f} (n = {len(x)})")
    for st, sub in x.groupby(x.session.str.contains("PostMPTP")):
        r2, p2 = stats.spearmanr(sub.latency, sub[col])
        print(f"   {'post' if st else 'pre'} only: rho = {r2:.3f}, p = {p2:.4f} (n = {len(sub)})")

**Result: blocked (methods underspecified)**, as the selection of responses is (see `false-positive-rate`). Over sessions, mean PTN latency does not reach the paper's
threshold with velocity (rho = -0.22, p = 0.080) and does with the triceps longus 1 m2 (rho =
-0.32, p = 0.011, below 0.0125); within each state neither correlates (p 0.36 to 0.997).

### `trial-correlations`

> Trial by trial, PTN response latencies did not correlate with the peak velocity or the m2 EMG
> response (Spearman |rho| < 0.34, p > 0.05).

Blocked: the Methods define response onset on the trial-averaged SDF and give no way of measuring a
latency on a single trial, which this test needs.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| `cell-counts` | reproduced |
| `monkey-l-perturbations` | blocked: data missing |
| `baseline-rate` | reproduced |
| `ptn-faster-than-csn` | reproduced |
| `antidromic-latency` | partially reproduced |
| `false-positive-rate` | blocked: methods underspecified |
| `torque-related-fractions` | blocked: methods underspecified |
| `prevalence-unchanged` | blocked: methods underspecified |
| `ptn-more-responsive` | blocked: methods underspecified |
| `one-direction-half` | blocked: methods underspecified |
| `excitatory-majority` | blocked: methods underspecified |
| `magnitude-unchanged` | blocked: methods underspecified |
| `venus-response-magnitudes` | blocked: methods underspecified |
| `latency-shorter` | blocked: methods underspecified |
| `latency-not-training` | blocked: methods underspecified |
| `fwhm-shorter-ptn` | blocked: methods underspecified |
| `dsi-reduced` | blocked: methods underspecified |
| `venus-dsi` | blocked: methods underspecified |
| `dsi-not-baseline` | blocked: methods underspecified |
| `kinematics-slower` | not reproduced |
| `kinematic-latency-unchanged` | not reproduced |
| `emg-m2-larger` | reproduced |
| `emg-duration-longer` | partially reproduced |
| `emg-onset-unchanged` | not reproduced |
| `emg-m1-weak` | partially reproduced |
| `velocity-matched-magnitude` | blocked: methods underspecified |
| `session-correlations` | blocked: methods underspecified |
| `trial-correlations` | blocked: methods underspecified |

**How the analysis differs from the paper's.** Only monkey V is analysed, as the monkey L files
mark no torque perturbation; the paper pools both monkeys except in its supplementary table. The
response detection is one reading of the Methods (a threshold on the trial-averaged SDF set from
the baseline's SD), with a t-test across trials as a variant. Its false-positive rate over the
200 ms epoch is about 0.2, not the 0.01 the paper's threshold implies, so the claims that rest on
the selection of responses are blocked, though their analyses ran. With this detection the
latency shortening after MPTP appears with the paper's monkey V means, but the response magnitudes
fall after MPTP, which the paper did not find. The kinematic and EMG measures follow the Methods
where they are given; the response windows, the EMG threshold and the peak velocity window are
choices made here. The analog streams are realigned to the trials table.

**What was found about the dandiset.**

- The monkey L files mark no torque perturbation, so the monkey L half of the paper (115 and 73
  cells) cannot be tested; the Leu torque command is NaN in every pre-MPTP file, and 15 post-MPTP
  Leu files have no analog data.
- The analog streams (angle, torque command, EMG) are stored on a clock that falls about 5 ms per
  trial behind the trials table and the spikes, reaching hundreds of ms; realigned per trial, the
  torque command steps 4 to 7 ms after every labelled onset.
- The torque command's pulse is about four times larger in most post-MPTP sessions than in most
  pre-MPTP sessions (183 against 46 units, with 91 in some sessions of both), and post-MPTP pulses
  are followed by an opposite swing; the paper describes one 0.1 Nm, 50 ms impulse. The command's
  scale is not given, so this may be a change of torque or of the command's recording; the
  kinematic and EMG results differ between the sessions of different command sizes.
- The angle's reference frame says positive is extension; the data show positive is flexion. The
  series are named "Elbow" also for monkey L, whose task used the wrist.
- Four files repeat a block of another file's recording; in two of them the same unit, with
  identical spikes, is stored twice with different projection types.
- `unit_also_in_session_id` uses prefixes ("V", "L") that differ from the session ids, mostly
  names files that are not in the dandiset, and where the file is present it points to a
  different unit.
- The monkey L analog streams are at 100 Hz (the paper says 200 Hz). The dandiset description
  says MPTP was systemic (the files and the paper say intracarotid), that the task used the elbow
  in both monkeys, and that Leu was recorded in 1993 (some files are from 1992).
- The torque onset comes 1.70 to 1.73 s after the center target appears in 90% of perturbations,
  where the Methods describe an onset 1 to 2 s after center capture, at random.
- The monkey V cells with at least 5 perturbations per direction match the paper's monkey V counts
  of cells, PTNs and CSNs exactly, so the selection of cells can be recovered.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")